In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ジョブの監視またはキャンセル

ワークロードの一覧は、[ワークロードページ](https://quantum.cloud.ibm.com/workloads)で確認できます。

## ジョブのステータスを確認する
[ワークロードテーブル](https://quantum.cloud.ibm.com/workloads)を開き、「Status」列でジョブが完了したか失敗したかを確認してください。

## 残り使用量を確認する
[インスタンステーブル](https://quantum.cloud.ibm.com/instances)を開き、残り使用量を確認したいプランに対応するタブを選択してください。プランの合計使用時間と残り時間が表示されます。

## 送信されたジョブおよびワークロード数のメトリクスを確認する
[アナリティクスページ](https://quantum.cloud.ibm.com/analytics)にアクセスすると、送信されたジョブの総数、バッチワークロードとセッションワークロードの件数を確認できます。アナリティクスページは、ご自身が所有または管理しているアカウントのみ表示可能です。

## ジョブを監視する
ジョブインスタンスを使用して、適切なコマンドを呼び出すことでジョブのステータスを確認したり、結果を取得したりできます。

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | ジョブ完了直後に結果を確認します。ジョブの結果はジョブ完了後に利用可能になります。そのため、`job.result()` はジョブが完了するまでブロッキング呼び出しとなります。                               |
| job.job\_id()                 | そのジョブを一意に識別するIDを返します。後でジョブの結果を取得するにはジョブIDが必要です。そのため、後で取得する可能性のあるジョブのIDは保存しておくことをお勧めします。 |
| job.status()                  | ジョブのステータスを確認します。                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | 以前に送信したジョブを取得します。この呼び出しにはジョブIDが必要です。                                                                                                                                      |

<span id="retrieve-later"></span>
## 後でジョブ結果を取得する
`service.job(\<job\_id>)` を呼び出すことで、以前に送信したジョブを取得できます。ジョブIDが手元にない場合、または複数のジョブを一度に取得したい場合（廃止されたQPU（量子処理ユニット）のジョブを含む）は、代わりにオプションフィルターを指定して `service.jobs()` を呼び出してください。詳細は [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs) を参照してください。

> **Note:** `service.jobs()` は、非推奨の `qiskit-ibm-provider` パッケージで実行されたジョブも返します。旧来の（同様に非推奨の）`qiskit-ibmq-provider` パッケージで送信されたジョブは、もはや利用できません。

### 例
この例では、`my_backend` で実行された最新の10件のランタイムジョブを返します。

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## ジョブをキャンセルする
IBM Quantum Platformダッシュボードのワークロードページまたは特定のワークロードの詳細ページからジョブをキャンセルできます。ワークロードページでは、該当ワークロードの行末にあるオーバーフローメニューをクリックし、「Cancel」を選択してください。特定のワークロードの詳細ページにいる場合は、ページ上部の「Actions」ドロップダウンを使用して「Cancel」を選択してください。

Qiskitでは、`job.cancel()` を使用してジョブをキャンセルできます。

<span id="execution-spans"></span>
## Samplerの実行スパンを確認する
Qiskit Runtimeで実行された [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) ジョブの結果には、メタデータに実行タイミング情報が含まれています。
このタイミング情報を使用することで、特定のショットがQPU上でいつ実行されたかについて、上限と下限のタイムスタンプ境界を設定できます。
ショットは [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span) オブジェクトにグループ化されており、各オブジェクトは開始時刻、終了時刻、およびそのスパン内で収集されたショットの仕様を示します。

実行スパンは、[`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask) メソッドを提供することで、そのウィンドウ内に実行されたデータを指定します。このメソッドは、任意の [Primitive Unified Block（PUB）](/guides/primitive-input-output) インデックスを与えると、そのウィンドウ内に実行されたすべてのショットに対して `True` となるブールマスクを返します。PUBはSamplerのrun呼び出しに渡された順番でインデックスが付けられます。たとえば、PUBのシェイプが `(2, 3)` で4ショット実行された場合、マスクのシェイプは `(2, 3, 4)` となります。詳細は [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) APIページを参照してください。

例：
実行スパン情報を確認するには、`SamplerV2` が返す結果のメタデータを参照してください。このメタデータは `ExecutionSpans` オブジェクトの形式で提供されます。このオブジェクトはリスト型のコンテナで、`SliceSpan` などの `ExecutionSpan` サブクラスのインスタンスを格納しています。